# 08 · Time Spread & Storage — Deep Dive

Analysis of TTF calendar spreads (M-N vs M-M) as a function of EU storage levels.

**Key questions:**
- Does the Oct–Apr spread predict winter stress?
- At what fill level does the market switch from contango to backwardation?
- How does the Summer–Winter spread relate to injection pace?

**Configuration:** set `ANALYSIS_DATE` and `START_DATE` in cell 0.
---

In [ ]:
import sys, os, warnings
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from datetime import date
import calendar
warnings.filterwarnings('ignore')

for _c in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (_c / 'src' / 'agsi_client').exists():
        sys.path.insert(0, str(_c)); os.chdir(_c)
        print(f"✅ Root: {_c}"); break

# ═══════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════
ANALYSIS_DATE = None          # None = last data point | or date(2023,9,1)
START_DATE    = "2020-01-01"  # historical window for charts
# ═══════════════════════════════════════════════════════════════════

# ── Storage ───────────────────────────────────────────────────────
df_eu_full = pd.read_parquet("data/processed/eu_aggregate_full.parquet")
df_eu_full.index = pd.to_datetime(df_eu_full.index).tz_localize(None)
df_eu_full = df_eu_full.sort_index()

# ── TTF ───────────────────────────────────────────────────────────
ttf = pd.read_csv("data/raw/ttf_curve.csv", index_col=0, parse_dates=True)
ttf.index = pd.to_datetime(ttf.index).tz_localize(None)
ttf.columns = [c.strip() for c in ttf.columns]
for col in ttf.columns:
    ttf[col] = pd.to_numeric(ttf[col], errors='coerce')
ttf = ttf.sort_index()

# ── Apply date filters ─────────────────────────────────────────────
if ANALYSIS_DATE is None:
    ANALYSIS_DATE = ttf.index[-1].date()

df_eu = df_eu_full[
    (df_eu_full.index >= START_DATE) &
    (df_eu_full.index <= pd.Timestamp(ANALYSIS_DATE))
].copy()
ttf_w = ttf[
    (ttf.index >= START_DATE) &
    (ttf.index <= pd.Timestamp(ANALYSIS_DATE))
].copy()

fill_pct    = float(df_eu['gasInStorage'].dropna().iloc[-1] /
                    df_eu['workingGasVolume'].dropna().iloc[-1] * 100)
current_gwh = float(df_eu['gasInStorage'].dropna().iloc[-1]) * 1000
capacity    = float(df_eu['workingGasVolume'].dropna().iloc[-1]) * 1000

print(f"✅ Storage : {len(df_eu)} rows | {df_eu.index.min().date()} → {df_eu.index.max().date()}")
print(f"✅ TTF     : {len(ttf_w)} rows | {ttf_w.index.min().date()} → {ttf_w.index.max().date()}")
print(f"   Analysis date : {ANALYSIS_DATE}")
print(f"   Fill rate     : {fill_pct:.1f}%")
print(f"   Latest M1     : €{ttf_w['M1'].dropna().iloc[-1]:.2f}/MWh")

## 1 · Build Calendar Spreads with Real Month Labels

Instead of M+1/M+2, we label spreads by actual delivery months
(e.g. *Jul'26–Oct'26*, *Oct'26–Apr'27*).
This makes it immediately readable without mentally converting offsets.

In [ ]:
def month_label(base_date, offset):
    """Return 'MonYY' label for the month that is `offset` months from base_date.
    offset=1 → M1 (next month), offset=2 → M2, etc.
    """
    dm = ((base_date.month + offset - 1) % 12) + 1
    yr = base_date.year + (base_date.month + offset - 1) // 12
    return f"{calendar.month_abbr[dm]}'{str(yr)[-2:]}"

def build_spreads(ttf_df):
    """
    For each row (date), compute key calendar spreads using real month names.
    Returns a DataFrame with one row per date and columns named by actual months.
    """
    rows = []
    for dt, row in ttf_df.iterrows():
        r = {'date': dt}

        # ── Near-dated spreads (M+1 to M+N rolling) ──────────────────────────
        for n in range(2, 13):
            col_near = f'M{n-1}'; col_far = f'M{n}'
            if col_near in row and col_far in row:
                v_near = row[col_near]; v_far  = row[col_far]
                if not (pd.isna(v_near) or pd.isna(v_far)):
                    label = f"{month_label(dt,n-1)}–{month_label(dt,n)}"
                    r[f'spread_m{n-1}_m{n}'] = round(float(v_near) - float(v_far), 3)
                    r[f'label_m{n-1}_m{n}']  = label

        # ── Seasonal aggregates ───────────────────────────────────────────────
        def avg_months(offsets):
            vals = [float(row[f'M{m}']) for m in offsets
                    if f'M{m}' in row.index and not pd.isna(row[f'M{m}'])]
            return round(np.mean(vals), 3) if vals else np.nan

        # Summer = Apr–Sep (offsets depend on current month)
        # Find which offsets land in Apr-Sep
        summer_offsets = [m for m in range(1,13)
                          if (dt.month+m-1)%12+1 in range(4,10)]
        winter_offsets = [m for m in range(1,13)
                          if (dt.month+m-1)%12+1 in [10,11,12,1,2,3]]
        q_groups = {q: [m for m in range(1,25)
                         if ((dt.month+m-1)%12)//3+1 == q and m <= 12]
                    for q in range(1,5)}

        r['summer_avg'] = avg_months(summer_offsets[:6])
        r['winter_avg'] = avg_months(winter_offsets[:6])
        r['sw_spread']  = round(r['winter_avg'] - r['summer_avg'], 3)                           if not (pd.isna(r['summer_avg']) or pd.isna(r['winter_avg'])) else np.nan

        # Q labels with actual year
        for q in range(1,5):
            vals = avg_months(q_groups[q])
            r[f'Q{q}_avg'] = vals
            # Label: QN'YY where YY from first month in group
            if q_groups[q]:
                m0 = q_groups[q][0]
                dm0 = (dt.month+m0-1)%12+1
                yr0 = dt.year + (dt.month+m0-1)//12
                r[f'Q{q}_label'] = f"Q{q}'{str(yr0)[-2:]}"

        # M1–M6 spread (injection premium)
        r['m1_m6_spread'] = round(float(row['M1'])-float(row['M6']), 3)                              if 'M1' in row and 'M6' in row                              and not (pd.isna(row['M1']) or pd.isna(row['M6'])) else np.nan

        # M1–M12 (full curve slope)
        r['m1_m12_spread'] = round(float(row['M1'])-float(row['M12']), 3)                               if 'M1' in row and 'M12' in row                               and not (pd.isna(row['M1']) or pd.isna(row['M12'])) else np.nan

        rows.append(r)

    return pd.DataFrame(rows).set_index('date')

spreads = build_spreads(ttf_w)

# Merge with storage
merged = spreads.join(df_eu[['full','injection','withdrawal','gasInStorage']], how='inner').dropna(subset=['full'])
print(f"✅ Spreads computed: {spreads.shape}")
print(f"✅ Merged with storage: {merged.shape}")
print(f"\nLatest spreads ({merged.index[-1].date()}):")
print(f"  M1–M6  spread : €{merged['m1_m6_spread'].iloc[-1]:.2f}/MWh")
print(f"  M1–M12 spread : €{merged['m1_m12_spread'].iloc[-1]:.2f}/MWh")
print(f"  W–S spread    : €{merged['sw_spread'].iloc[-1]:.2f}/MWh")

## 2 · Spread History — Real Month Labels

Key spreads over time. Hover to see exact month names (e.g. Jul'26–Oct'26).
Positive = near month more expensive = backwardation (tight supply).

In [ ]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=[
        "M1–M12 Spread (full curve slope, €/MWh)",
        "M1–M6 Spread (near-term premium, €/MWh)",
        "Winter–Summer Spread (seasonal, €/MWh)",
    ],
    row_heights=[0.35,0.35,0.30])

def spread_trace(s, name, color, row):
    fig.add_trace(go.Scatter(x=s.index, y=s, mode='lines', name=name,
        line=dict(color=color, width=1.5), fill='tozeroy',
        fillcolor=color.replace(')',',0.10)').replace('rgb','rgba')
                 if 'rgb' in color else f'rgba(46,117,182,0.10)'),
    row=row, col=1)
    fig.add_hline(y=0, line_color='black', line_width=0.8, row=row, col=1)

spread_trace(merged['m1_m12_spread'], 'M1–M12', '#2E75B6', 1)
spread_trace(merged['m1_m6_spread'],  'M1–M6',  '#ff7f0e', 2)
spread_trace(merged['sw_spread'],     'W–S',    '#d62728', 3)
fig.add_hline(y=5, line_dash='dot', line_color='green', row=3, col=1,
              annotation_text='Injection breakeven €5')

fig.update_layout(template='plotly_white', height=600,
                  hovermode='x unified', showlegend=False)
fig.show()

print(f"\nCurrent levels ({merged.index[-1].date()}):")
print(f"  M1–M12 : €{merged['m1_m12_spread'].iloc[-1]:.2f}/MWh  "
      f"({'backwardation' if merged['m1_m12_spread'].iloc[-1]>0 else 'contango'})")
print(f"  M1–M6  : €{merged['m1_m6_spread'].iloc[-1]:.2f}/MWh")
print(f"  W–S    : €{merged['sw_spread'].iloc[-1]:.2f}/MWh")

## 3 · Spread vs. Storage Fill Rate

Does the spread predict when the market prices winter scarcity?
We look for the **inflection point** — the fill level where the
W–S spread crosses from negative to positive (market starts pricing winter).

In [ ]:
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

fig = make_subplots(rows=1, cols=3,
    subplot_titles=[
        "W–S Spread vs Fill %",
        "M1–M12 Spread vs Fill %",
        "M1–M6 Spread vs Fill %",
    ])

spread_pairs = [
    ('sw_spread',     'W–S Spread',  '#d62728'),
    ('m1_m12_spread', 'M1–M12',      '#2E75B6'),
    ('m1_m6_spread',  'M1–M6',       '#ff7f0e'),
]

for col_idx, (spread_col, label, color) in enumerate(spread_pairs, 1):
    data = merged[['full', spread_col]].dropna()
    x    = data['full'].values
    y    = data[spread_col].values

    # Scatter coloured by year
    for year in sorted(data.index.year.unique()):
        sub = data[data.index.year == year]
        fig.add_trace(go.Scatter(
            x=sub['full'], y=sub[spread_col],
            mode='markers', name=str(year),
            marker=dict(size=3, opacity=0.5),
            legendgroup=str(year),
            showlegend=(col_idx==1),
        ), row=1, col=col_idx)

    # Polynomial fit (degree 2)
    x_range = np.linspace(x.min(), x.max(), 100)
    poly    = PolynomialFeatures(degree=2)
    X_fit   = poly.fit_transform(x.reshape(-1,1))
    model   = LinearRegression().fit(X_fit, y)
    y_fit   = model.predict(poly.transform(x_range.reshape(-1,1)))
    r2      = model.score(X_fit, y)

    fig.add_trace(go.Scatter(x=x_range, y=y_fit, mode='lines', name=f'Fit (R²={r2:.2f})',
        line=dict(color='black', dash='dash', width=2), showlegend=(col_idx==1)),
        row=1, col=col_idx)

    # Zero-crossing (inflection point)
    sign_changes = np.where(np.diff(np.sign(y_fit)))[0]
    for sc in sign_changes:
        x_cross = x_range[sc]
        fig.add_vline(x=x_cross, line_dash='dot', line_color='green',
                      line_width=1.5, row=1, col=col_idx)
        fig.add_annotation(x=x_cross, y=y_fit.max()*0.9,
                           text=f"~{x_cross:.0f}%", showarrow=False,
                           font=dict(size=9, color='green'), row=1, col=col_idx)

    fig.add_hline(y=0, line_color='black', line_width=0.8, row=1, col=col_idx)

    # Current point
    cx = fill_pct
    cy = float(merged[spread_col].dropna().iloc[-1])
    fig.add_trace(go.Scatter(x=[cx], y=[cy], mode='markers',
        marker=dict(size=12, color='red', symbol='star',
                    line=dict(color='white',width=1.5)),
        name=f'Today ({cx:.1f}%)', showlegend=(col_idx==1)),
        row=1, col=col_idx)

fig.update_xaxes(title_text="EU Fill Rate (%)", ticksuffix="%")
fig.update_yaxes(title_text="Spread (€/MWh)", row=1, col=1)
fig.update_layout(template='plotly_white', height=480,
                  legend=dict(orientation='v', x=1.01))
fig.show()

print("\nInflection points (fill % where spread ≈ 0):")
for spread_col, label, _ in spread_pairs:
    data = merged[['full', spread_col]].dropna()
    x = data['full'].values; y = data[spread_col].values
    poly = PolynomialFeatures(degree=2)
    model = LinearRegression().fit(poly.fit_transform(x.reshape(-1,1)), y)
    x_range = np.linspace(x.min(), x.max(), 1000)
    y_fit = model.predict(poly.transform(x_range.reshape(-1,1)))
    crosses = x_range[np.where(np.diff(np.sign(y_fit)))[0]]
    r2 = model.score(poly.fit_transform(x.reshape(-1,1)), y)
    cross_str = ', '.join(f'{c:.1f}%' for c in crosses) if len(crosses) else 'none'
    print(f"  {label:<14}: zero at fill={cross_str}  R²={r2:.3f}")

## 4 · Spread Regimes vs. Storage Quintiles

We split historical data into 5 fill-rate buckets (quintiles) and show
the distribution of spreads in each. This reveals the non-linearity —
how sharply spreads move when fill drops below a critical threshold.

In [ ]:
data = merged[['full','sw_spread','m1_m12_spread','m1_m6_spread']].dropna()
data['fill_quintile'] = pd.qcut(data['full'], q=5,
    labels=['Q1\n(lowest\nfill)','Q2','Q3','Q4','Q5\n(highest\nfill)'])

fig = make_subplots(rows=1, cols=3,
    subplot_titles=['W–S Spread by Fill Quintile',
                    'M1–M12 Spread by Fill Quintile',
                    'M1–M6 Spread by Fill Quintile'])

spread_cols = [('sw_spread','#d62728'),('m1_m12_spread','#2E75B6'),('m1_m6_spread','#ff7f0e')]

for col_idx, (spread_col, color) in enumerate(spread_cols, 1):
    for q_label in data['fill_quintile'].cat.categories:
        sub = data[data['fill_quintile']==q_label][spread_col].dropna()
        fig.add_trace(go.Box(
            y=sub, name=str(q_label),
            marker_color=color, opacity=0.7,
            boxmean=True, showlegend=False,
        ), row=1, col=col_idx)
    fig.add_hline(y=0, line_color='black', line_width=0.8, row=1, col=col_idx)
    fig.add_hline(y=5, line_dash='dot', line_color='green', row=1, col=col_idx)

fig.update_yaxes(title_text="Spread (€/MWh)", row=1, col=1)
fig.update_layout(template='plotly_white', height=450)
fig.show()

print("Median spread by fill quintile:")
print(f"{'Quintile':<25} {'W–S':>8} {'M1–M12':>8} {'M1–M6':>8}  {'Fill range':>15}")
for q in data['fill_quintile'].cat.categories:
    sub = data[data['fill_quintile']==q]
    fr  = f"{sub['full'].min():.0f}–{sub['full'].max():.0f}%"
    print(f"  {str(q).replace(chr(10),' '):<23} "
          f"{sub['sw_spread'].median():>7.2f} "
          f"{sub['m1_m12_spread'].median():>8.2f} "
          f"{sub['m1_m6_spread'].median():>7.2f}  {fr:>15}")

## 5 · Interactive: Spread Curve at Any Date

Plots the full month-by-month spread curve (consecutive month spread M_n – M_{n+1})
at a chosen date. X-axis shows actual month names, not offsets.
Change `CURVE_DATE` to explore any historical date.

In [ ]:
CURVE_DATE = None  # None = latest | or '2022-09-01'

def plot_spread_curve(ttf_df, storage_df, plot_date=None):
    if plot_date is None:
        dt = pd.Timestamp(ttf_df.dropna(how='all').index[-1])
    else:
        dt = pd.Timestamp(plot_date)
        dt = ttf_df.index[ttf_df.index <= dt][-1]

    row  = ttf_df.loc[dt]
    fill = storage_df['full'].loc[storage_df.index <= dt].iloc[-1]            if (storage_df.index <= dt).any() else None

    x_labels, y_spreads, y_prices, colors = [], [], [], []
    season_c = {12:'#d62728',1:'#d62728',2:'#d62728',3:'#d62728',
                4:'#2ca02c',5:'#2ca02c',6:'#2ca02c',
                7:'#ff7f0e',8:'#ff7f0e',9:'#ff7f0e',
                10:'#9467bd',11:'#9467bd'}

    for m in range(1, 13):
        col_n = f'M{m}'; col_f = f'M{m+1}'
        if col_n not in row or col_f not in row: continue
        if pd.isna(row[col_n]) or pd.isna(row[col_f]): continue
        dm_n = ((dt.month+m-1)%12)+1;  dm_f = ((dt.month+m)%12)+1
        yr_n = dt.year+(dt.month+m-1)//12; yr_f = dt.year+(dt.month+m)//12
        label = f"{calendar.month_abbr[dm_n]}'{str(yr_n)[-2:]}–{calendar.month_abbr[dm_f]}'{str(yr_f)[-2:]}"
        x_labels.append(label)
        y_spreads.append(round(float(row[col_n])-float(row[col_f]), 3))
        y_prices.append(float(row[col_n]))
        colors.append(season_c[dm_n])

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
        subplot_titles=[
            f"Price level — {dt.strftime('%d %b %Y')}  (fill: {fill:.1f}%)" if fill else f"Price — {dt.date()}",
            "Month-on-month spread (near – far, €/MWh)  →  positive = backwardation",
        ], row_heights=[0.45,0.55])

    fig.add_trace(go.Bar(x=x_labels, y=y_prices,
        marker_color=colors, opacity=0.8, name='Price'),
        row=1, col=1)

    fig.add_trace(go.Bar(x=x_labels, y=y_spreads,
        marker_color=['#d62728' if v>0 else '#2E75B6' for v in y_spreads],
        opacity=0.85, name='Spread'),
        row=2, col=1)
    fig.add_hline(y=0, line_color='black', line_width=1, row=2, col=1)

    fig.update_yaxes(title_text="€/MWh", row=1, col=1)
    fig.update_yaxes(title_text="€/MWh spread", row=2, col=1)
    fig.update_xaxes(tickangle=45, tickfont=dict(size=8))
    fig.update_layout(template='plotly_white', height=550, showlegend=False,
        title=f"TTF Spread Structure — {dt.strftime('%d %b %Y')}")
    fig.show()

    total_slope = sum(y_spreads)
    print(f"  Cumulative M1→M12 slope : €{total_slope:.2f}/MWh  "
          f"({'backwardation' if total_slope>0 else 'contango'})")
    print(f"  Steepest spread         : {x_labels[np.argmax(np.abs(y_spreads))]} "
          f"€{max(y_spreads, key=abs):.2f}/MWh")

plot_spread_curve(ttf_w, df_eu, CURVE_DATE)

## 6 · Weekly Storage Change vs. Spread

Do spreads widen after unexpected storage draws?
We compute the week-on-week change in EU fill % and correlate
with the M1–M6 spread move over the same period.

In [ ]:
# Weekly changes
weekly = merged[['full','sw_spread','m1_m12_spread','m1_m6_spread']].resample('W').last().dropna()
weekly['fill_wow']    = weekly['full'].diff()
weekly['sw_wow']      = weekly['sw_spread'].diff()
weekly['m1m12_wow']   = weekly['m1_m12_spread'].diff()

fig = make_subplots(rows=1, cols=2,
    subplot_titles=[
        "Weekly Fill Change vs W–S Spread Change",
        "Weekly Fill Change vs M1–M12 Spread Change",
    ])

for col_idx, (y_col, label) in enumerate(
    [('sw_wow','W–S Spread Δ (€/MWh)'),('m1m12_wow','M1–M12 Spread Δ (€/MWh)')], 1):

    data_w = weekly[['fill_wow', y_col]].dropna()
    slope, intercept, r, p, _ = stats.linregress(data_w['fill_wow'], data_w[y_col])

    fig.add_trace(go.Scatter(
        x=data_w['fill_wow'], y=data_w[y_col],
        mode='markers',
        marker=dict(size=4, opacity=0.5, color='#2E75B6'),
        text=data_w.index.strftime('%d %b %Y'),
        hovertemplate='%{text}<br>Fill Δ:%{x:.2f}pp<br>Spread Δ:€%{y:.2f}<extra></extra>',
        showlegend=False,
    ), row=1, col=col_idx)

    # Regression line
    x_range = np.linspace(data_w['fill_wow'].min(), data_w['fill_wow'].max(), 50)
    fig.add_trace(go.Scatter(x=x_range, y=slope*x_range+intercept,
        mode='lines', line=dict(color='red', dash='dash', width=2),
        name=f"slope={slope:.3f}, R²={r**2:.2f}", showlegend=True),
        row=1, col=col_idx)

    fig.add_hline(y=0, line_color='black', line_width=0.5, row=1, col=col_idx)
    fig.add_vline(x=0, line_color='black', line_width=0.5, row=1, col=col_idx)

fig.update_xaxes(title_text="Weekly fill rate change (pp)", ticksuffix="pp")
fig.update_yaxes(title_text="Weekly spread change (€/MWh)", row=1, col=1)
fig.update_layout(template='plotly_white', height=440)
fig.show()

print("Regression: Weekly fill change → spread change")
for y_col, label in [('sw_wow','W–S'),('m1m12_wow','M1–M12')]:
    data_w = weekly[['fill_wow',y_col]].dropna()
    slope, _, r, p, _ = stats.linregress(data_w['fill_wow'], data_w[y_col])
    print(f"  {label}: slope={slope:.3f} €/MWh per pp fill  R²={r**2:.3f}  p={p:.4f}")
    print(f"    → 1pp unexpected fill draw = €{-slope:.2f}/MWh spread widening")

## 7 · Spread Heatmap — Month vs. Year

Shows the W–S spread for each calendar month and year.
Reveals seasonal patterns and the 2022 stress regime.

In [ ]:
heat = merged[['sw_spread']].copy()
heat['month'] = heat.index.month
heat['year']  = heat.index.year

pivot = heat.pivot_table(index='month', columns='year', values='sw_spread', aggfunc='mean')
pivot.index = [calendar.month_abbr[m] for m in pivot.index]

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=[str(y) for y in pivot.columns],
    y=pivot.index.tolist(),
    colorscale='RdBu',
    zmid=0,
    colorbar=dict(title='W–S Spread (€/MWh)'),
    hovertemplate='%{y} %{x}<br>W–S: €%{z:.2f}/MWh<extra></extra>',
))
fig.update_layout(
    title="Winter–Summer Spread Heatmap — Monthly Average by Year",
    xaxis_title="Year",
    yaxis_title="Month",
    template="plotly_white",
    height=420,
)
fig.show()

print("\nAverage W–S spread by month (all years):")
monthly_avg = heat.groupby('month')['sw_spread'].mean()
for m, v in monthly_avg.items():
    bar = '█' * int(abs(v)) + ('↑' if v>0 else '↓')
    print(f"  {calendar.month_abbr[m]}: {v:>+7.2f} €/MWh  {bar}")